# Vault RAG — Qdrant

## Co się zmienia względem lekcji 11

Lekcja 11 używała grafu vaultu (strony `Powierzchnie/`, `Kategorie/`) jako indeksu do wyszukiwania kandydatów.
Wymagało to utrzymania 307 stron powierzchni i 285 stron kategorii, a wyszukiwanie było czystym dopasowaniem napisów.

**Lekcja 13 zastępuje kroki 2 i 2.5 jednym zapytaniem Qdrant:**

```
Lekcja 11:  pytanie → [LLM] intencja → [Python] graf vaultu → [LLM] filtr BHP → [LLM] rekomendacja
Lekcja 13:  pytanie → [LLM] intencja → [Qdrant] filtr payload + wektor → [LLM] filtr BHP → [LLM] rekomendacja
```

### Jak działa Qdrant w tym pipeline'ie

Każdy z 262 produktów jest zaindeksowany jako punkt Qdrant:
- **Wektor**: embedding opisu produktu (`text-embedding-3-small`, 1536 wymiarów)
- **Payload**: metadane ze stron produktów (`kategorie`, `surfaces`, `metody`, `odradzane`)

Zapytanie łączy dwa sygnały:
1. **Filtr payload** (deterministyczny) — `kategorie` lub `surfaces` z intencji muszą pasować do pól payload
2. **Podobieństwo wektorowe** (semantyczne) — produkty rankowane są według bliskości do pytania

```
embed(pytanie) + Filter(kategorie in intent.categories OR surfaces in intent.surfaces)
→ top-20 produktów rankowanych przez cosinus
```

Fallback do czystego wyszukiwania semantycznego (bez filtra) gdy wynik < 3.

### Korzyści
- Wyszukiwanie semantyczne wychwytuje niezgodności słownikowe (np. `aluminiowe części maszyn` → `czyszczenie-aluminium`)
- Filtr payload zachowuje precyzję kategorii bez pełnego grafu vaultu
- Graf vaultu nie jest już potrzebny do wyszukiwania — tylko strony produktów
- Score Qdrant może zastąpić część logiki filtra BHP (odradzane w payload)

In [1]:
%pip install qdrant-client openai instructor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 40.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qdrant-client]0m [qdrant-client]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import re
import time
import json
from pathlib import Path
from datetime import datetime
from typing import List
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

from openai import OpenAI
from pydantic import BaseModel, Field
import instructor
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchAny,
)

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "Brak klucza")

LLM_MODEL   = "google/gemini-3.1-flash-lite"
EMBED_MODEL = "openai/text-embedding-3-small"  # via OpenRouter, 1536 dims
COLLECTION  = "zurada_products"

BASE_DIR       = Path(".")
VAULT_DIR      = BASE_DIR / "../11_vault_rag_improvements/zurada_vault"
QUESTIONS_FILE = BASE_DIR / "../5_navie_rag_extended_data/extended_golden_questions.json"
QDRANT_PATH    = BASE_DIR / "qdrant_storage"  # persistent across runs

_raw_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

# LLM client (structured output via instructor)
llm = instructor.from_openai(_raw_client, mode=instructor.Mode.JSON)

# Embedding client — same OpenRouter base, raw client (no instructor wrapping needed)
embed_client = _raw_client

# Qdrant — local file-based storage, persistent across notebook runs
qdrant = QdrantClient(path=str(QDRANT_PATH))

with open(QUESTIONS_FILE, encoding="utf-8") as f:
    questions = json.load(f)

print(f"LLM:      {LLM_MODEL}")
print(f"Embed:    {EMBED_MODEL}")
print(f"Vault:    {VAULT_DIR.resolve()}")
print(f"Qdrant:   {QDRANT_PATH.resolve()}")
print(f"Questions: {len(questions)}")
print(f"API key:   {'OK' if OPENROUTER_API_KEY != 'Brak klucza' else 'BRAK'}")

LLM:      google/gemini-3.1-flash-lite
Embed:    openai/text-embedding-3-small
Vault:    /Users/adamzurada/projects/ai-knowledge/rag-product-recommend/lessons/11_vault_rag_improvements/zurada_vault
Qdrant:   /Users/adamzurada/projects/ai-knowledge/rag-product-recommend/lessons/13_qdrant_rag/qdrant_storage
Questions: 9
API key:   OK


---
## MAP_KNOWLEDGE + modele Pydantic

Identyczne jak w lekcji 11.

In [2]:
MAP_KNOWLEDGE = """
Vault Zurada to baza wiedzy produktów czyszczących, zorganizowana jako mapa połączonych stron.

Struktura vaultu:
- Powierzchnie/  — strony dla każdej powierzchni (np. toalety, podłogi, szkło)
- Kategorie/     — strony dla każdej kategorii (np. wc, zele-do-wc, odkamienianie)
- Metody/        — metody mycia (np. ręczne, myjnia, pianowanie)
- Produkty/      — pełne opisy produktów (frontmatter: product_id, ph, metody, kategorie)

Zasady odpowiedzi:
- Odpowiadaj WYŁĄCZNIE na podstawie dostarczonych stron produktów
- Nie wymyślaj cech produktów spoza dostarczonych danych
- Odróżniaj formy (żel vs piana), przeznaczenie (domowe vs przemysłowe), metodę (ręczne vs maszynowe)
"""


class ProductCandidate(BaseModel):
    product_id: int
    reason: str = Field(description="Jedno zdanie uzasadnienia")


class RecommendModel(BaseModel):
    chain_of_thought: List[str]         = Field(default_factory=list)
    product_ids: List[ProductCandidate] = Field(default_factory=list)
    excluded_product_ids: List[ProductCandidate] = Field(default_factory=list)



print("MAP_KNOWLEDGE and models defined")

MAP_KNOWLEDGE and models defined


---
## Krok 0 — Parsowanie metadanych produktów z vaultu

Czytamy strony produktów raz, wyciągamy:
- `kategorie` i `metody` z frontmattera
- `surfaces` (dozwolone) i `odradzane` z linków wiki w treści
- tekst opisu do osadzenia

Te metadane trafią jako **payload** do Qdrant.

In [3]:
def parse_frontmatter_list(md_text: str, field: str) -> list[str]:
    m = re.search(rf'^{field}:\s*\[(.*)\]', md_text, re.MULTILINE)
    if not m:
        return []
    return [v.strip().strip('"') for v in m.group(1).split(',') if v.strip()]


def parse_frontmatter_field(md_text: str, field: str) -> str | None:
    m = re.search(rf'^{field}:\s*(.+)$', md_text, re.MULTILINE)
    return m.group(1).strip().strip('"') if m else None


def parse_wiki_links(md_text: str, prefix: str) -> list[str]:
    """Extract link targets under a given Powierzchnie/ or Odradzane/ prefix."""
    pattern = rf'\[\[{re.escape(prefix)}/([^|\]]+)[|\]]'
    return re.findall(pattern, md_text)


def product_embed_text(md_text: str, name: str) -> str:
    """Build the text that will be embedded for this product."""
    title = parse_frontmatter_field(md_text, "title") or name
    parts = md_text.split("---", 2)
    body  = parts[2] if len(parts) >= 3 else md_text
    # Keep only ## Opis and ## Właściwości sections
    body  = re.sub(r'^\*\*[^*]+\*\*.*$', '', body, flags=re.MULTILINE)
    body  = re.sub(r'\[\[.*?\]\]', '', body)
    body  = re.sub(r'#+\s+', '', body)
    body  = re.sub(r'[⛔>]', '', body)
    body  = ' '.join(body.split())
    return f"{title}. {body}"


# Build product metadata dict
products: dict[str, dict] = {}

for f in sorted((VAULT_DIR / "Produkty").glob("*.md")):
    text = f.read_text(encoding="utf-8")
    pid  = parse_frontmatter_field(text, "product_id")
    if not pid:
        continue
    products[f.stem] = {
        "product_id": int(pid),
        "name":       f.stem,
        "title":      parse_frontmatter_field(text, "title") or f.stem,
        "kategorie":  parse_frontmatter_list(text, "kategorie"),
        "metody":     parse_frontmatter_list(text, "metody"),
        "surfaces":   parse_wiki_links(text, "Powierzchnie"),
        "odradzane":  parse_wiki_links(text, "Odradzane"),
        "embed_text": product_embed_text(text, f.stem),
    }

print(f"Parsed {len(products)} products")
# Quick sanity check
sample = list(products.values())[0]
print(f"Sample: {sample['name']}")
print(f"  kategorie: {sample['kategorie'][:3]}")
print(f"  surfaces:  {sample['surfaces'][:3]}")
print(f"  odradzane: {sample['odradzane'][:3]}")
print(f"  embed_text[:120]: {sample['embed_text'][:120]}")

Parsed 262 products
Sample: Zurada Agro Kompleks
  kategorie: ['chemia-do-maszyn-rolniczych', 'maszyny-rolnicze', 'mycie-maszyn']
  surfaces:  ['lakier samochodowy', 'metal', 'tworzywa sztuczne']
  odradzane: []
  embed_text[:120]: Zestaw do mycia maszyn. Zurada Agro Kompleks Zestaw do czyszczenia, odtłuszczania i zabezpieczania maszyn rolniczych. --


In [10]:
# Preview: what exactly goes into Qdrant for one product
import random

sample_name = "Zurada WC Żel Dom"  # change to any product name
p = products[sample_name]

print("=" * 70)
print(f"PRODUKT: {p['name']}")
print("=" * 70)

print("\n--- TEKST DO OSADZENIA (embed_text) ---")
print(p["embed_text"])

print("\n--- PAYLOAD (metadane w Qdrant) ---")
payload = {
    "name":      p["name"],
    "title":     p["title"],
    "kategorie": p["kategorie"],
    "surfaces":  p["surfaces"],
    "metody":    p["metody"],
    "odradzane": p["odradzane"],
}
for key, val in payload.items():
    print(f"  {key:12s}: {val}")

PRODUKT: Zurada WC Żel Dom

--- TEKST DO OSADZENIA (embed_text) ---
Żel do WC. Zurada WC Żel Dom Gęsty żel do czyszczenia toalet i ceramiki sanitarnej, usuwa kamień, rdzę oraz osady. --- - - - --- Opis Zurada Sanit Żel to skuteczny preparat do czyszczenia muszli WC oraz innych urządzeń sanitarnych. Gęsta formuła dobrze przylega do pionowych powierzchni, ułatwiając usuwanie kamienia, rdzy i osadów z twardej wody. Produkt pozostawia świeży, kwiatowy zapach i pomaga utrzymać higieniczną czystość w łazience. Przeznaczony jest do stosowania bezpośredniego, szczególnie w miejscach narażonych na osady mineralne. Właściwości Preparat działa na typowe zabrudzenia sanitarne, w tym kamień, rdzę i uporczywe naloty. Dzięki żelowej konsystencji dłużej utrzymuje się na czyszczonej powierzchni, zwiększając skuteczność działania.

--- PAYLOAD (metadane w Qdrant) ---
  name        : Zurada WC Żel Dom
  title       : Żel do WC
  kategorie   : ['zele-do-wc', 'wc', 'lazienka', 'czyszczenie-sanitarne', 'odk

---
## Krok 1 — Indeksowanie produktów w Qdrant

Tworzymy kolekcję Qdrant i indeksujemy wszystkie 262 produkty.
Kolekcja jest trwała (`qdrant_storage/`) — ten krok uruchamia się tylko raz.

Każdy punkt Qdrant:
- `id`: product_id (int)
- `vector`: embedding opisu produktu (1536 dims)
- `payload`: kategorie, surfaces, metody, odradzane, name, title

In [4]:
def embed(texts: list[str]) -> list[list[float]]:
    """Batch embed texts using OpenAI text-embedding-3-small."""
    resp = embed_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [e.embedding for e in resp.data]


def embed_one(text: str) -> list[float]:
    return embed([text])[0]


# Create collection if it doesn't exist yet
existing = [c.name for c in qdrant.get_collections().collections]

if COLLECTION not in existing:
    print(f"Creating collection '{COLLECTION}'...")
    qdrant.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
    )

    # Embed all products in batches of 50
    product_list = list(products.values())
    BATCH = 50
    points = []

    for i in range(0, len(product_list), BATCH):
        batch = product_list[i : i + BATCH]
        vectors = embed([p["embed_text"] for p in batch])
        for p, vec in zip(batch, vectors):
            points.append(PointStruct(
                id      = p["product_id"],
                vector  = vec,
                payload = {
                    "name":      p["name"],
                    "title":     p["title"],
                    "kategorie": p["kategorie"],
                    "surfaces":  p["surfaces"],
                    "metody":    p["metody"],
                    "odradzane": p["odradzane"],
                },
            ))
        print(f"  Embedded {min(i + BATCH, len(product_list))}/{len(product_list)}")

    qdrant.upsert(collection_name=COLLECTION, points=points)
    print(f"Indexed {len(points)} products into '{COLLECTION}'")
else:
    count = qdrant.count(collection_name=COLLECTION).count
    print(f"Collection '{COLLECTION}' already exists ({count} points) — skipping indexing")

Collection 'zurada_products' already exists (262 points) — skipping indexing


---
## Krok 2 — Ekstrakcja intencji

Taki sam jak w lekcji 11 — LLM zwraca kategorie i powierzchnie jako closed-set enum.
Te wartości będą użyte jako filtr payload w Qdrant.

In [5]:
import enum as _enum


def _make_enum(name: str, keys: list[str]) -> type:
    return _enum.Enum(
        name,
        {k.upper().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", ""): k
         for k in sorted(keys)},
        type=str,
    )


# Build enums from vault pages (needed to constrain intent extraction)
def _vault_page_names(subdir: str) -> list[str]:
    return [f.stem for f in (VAULT_DIR / subdir).glob("*.md")]


SurfaceEnum  = _make_enum("SurfaceEnum",  _vault_page_names("Powierzchnie"))
CategoryEnum = _make_enum("CategoryEnum", _vault_page_names("Kategorie"))
MethodEnum   = _make_enum("MethodEnum",   _vault_page_names("Metody"))


class IntentModel(BaseModel):
    surfaces: List[SurfaceEnum] = Field(
        default_factory=list,
        description="Powierzchnie fizyczne wymienione lub sugerowane w pytaniu. Domyślnie pusta lista.",
    )
    categories: List[CategoryEnum] = Field(
        default_factory=list,
        description="Kategorie produktów pasujące do pytania. Domyślnie pusta lista.",
    )
    methods: List[MethodEnum] = Field(
        default_factory=list,
        description="Metody aplikacji — tylko gdy pytanie jawnie wskazuje metodę. Domyślnie pusta lista.",
    )


INTENT_SYSTEM = """Jesteś analitykiem zapytań dla bazy wiedzy produktów czyszczących Zurada.

{MAP_KNOWLEDGE}

Na podstawie pytania klienta zidentyfikuj, które kategorie, powierzchnie i metody są potrzebne.
ZASADY:
1. UŻYWAJ WYŁĄCZNIE WARTOŚCI Z LISTY — nie wymyślaj własnych.
2. ZACHOWAJ ORYGINALNĄ PISOWNIĘ Z ENUMÓW.
3. 'surfaces' to fizyczne obiekty; 'categories' to grupy produktów.
Wybierz do 6 pozycji per pole."""


def extract_intent(question: str) -> tuple[IntentModel, object]:
    intent, completion = llm.chat.completions.create_with_completion(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": INTENT_SYSTEM.format(MAP_KNOWLEDGE=MAP_KNOWLEDGE)},
            {"role": "user",   "content": question},
        ],
        response_model=IntentModel,
        max_tokens=2000,
        temperature=0,
    )
    return intent, completion.usage


print(f"SurfaceEnum:  {len(list(SurfaceEnum))} values")
print(f"CategoryEnum: {len(list(CategoryEnum))} values")
print(f"MethodEnum:   {len(list(MethodEnum))} values")

# Demo
demo_q = questions[0]
intent_demo, _ = extract_intent(demo_q["question"])
print(f"\nDemo question: {demo_q['question']}")
print(f"Surfaces:   {[s.value for s in intent_demo.surfaces]}")
print(f"Categories: {[c.value for c in intent_demo.categories]}")

SurfaceEnum:  307 values
CategoryEnum: 285 values
MethodEnum:   22 values

Demo question: Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę.
Surfaces:   ['toalety']
Categories: ['zele-do-wc', 'higiena-toalet', 'odkamienianie', 'dla-domu']


---
## Krok 3 — Wyszukiwanie kandydatów przez Qdrant

Zastępuje kroki 2 i 3 z lekcji 11 (graf vaultu + ładowanie stron).

Jedno zapytanie Qdrant łączy:
- **Filtr payload** (`should` = OR): kategorie lub powierzchnie z intencji muszą pasować
- **Ranking wektorowy**: produkty sortowane według podobieństwa do pytania

Fallback do czystego wyszukiwania semantycznego gdy filtr daje < 3 wyników.

In [6]:
def load_product_pages(names: list[str]) -> str:
    pages = []
    for name in names:
        path = VAULT_DIR / "Produkty" / f"{name}.md"
        if path.exists():
            pages.append(path.read_text(encoding="utf-8"))
    return "\n\n---\n\n".join(pages)


def qdrant_retrieve(
    question: str,
    intent: IntentModel,
    top_k: int = 20,
) -> tuple[str, list[str], list[float]]:
    """Semantic search with payload filter from intent.
    Returns (product_context, product_names, scores).
    """
    q_vector = embed_one(question)

    # Build OR filter: kategorie match OR surfaces match
    conditions = []
    if intent.categories:
        conditions.append(FieldCondition(
            key="kategorie",
            match=MatchAny(any=[c.value for c in intent.categories]),
        ))
    if intent.surfaces:
        conditions.append(FieldCondition(
            key="surfaces",
            match=MatchAny(any=[s.value for s in intent.surfaces]),
        ))

    qdrant_filter = Filter(should=conditions) if conditions else None

    response = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_vector,
        query_filter=qdrant_filter,
        limit=top_k,
        with_payload=True,
    )

    # Fallback: drop filter when too few results
    if len(response.points) < 3 and qdrant_filter is not None:
        response = qdrant.query_points(
            collection_name=COLLECTION,
            query=q_vector,
            limit=top_k,
            with_payload=True,
        )

    names  = [r.payload["name"] for r in response.points]
    scores = [round(r.score, 4) for r in response.points]
    return load_product_pages(names), names, scores


# Demo
context, names, scores = qdrant_retrieve(demo_q["question"], intent_demo)
print(f"Retrieved {len(names)} candidates:")
for name, score in zip(names, scores):
    pid = products.get(name, {}).get("product_id", "?")
    print(f"  [{pid:3}] {score:.4f}  {name}")

Retrieved 20 candidates:
  [139] 0.6460  Zurada WC Żel Dom
  [ 79] 0.6328  Zurada Biel Moc
  [ 12] 0.6269  Zurada Sanit Żel
  [ 82] 0.6245  Zurada Toaleta Żel
  [ 80] 0.6186  Zurada Kamień Gastro
  [129] 0.5858  Zurada Szkło Klar
  [133] 0.5726  Zurada Kamień Łazienka
  [  2] 0.5664  Zurada Szklany Blask
  [ 29] 0.5640  Zurada Sanitar Błysk
  [ 68] 0.5617  Zurada Kwasowa Moc
  [229] 0.5601  Zurada Srebrny Blask
  [  4] 0.5599  Zurada Kuchnia Czysta
  [ 41] 0.5586  Zurada Kryształ
  [ 11] 0.5584  Zurada Czysta Łazienka
  [ 33] 0.5463  Zurada Kwas Pro
  [ 46] 0.5404  Zurada Posadzka Lux
  [ 39] 0.5403  Zurada Szkło Błysk
  [ 92] 0.5386  Zurada Odkamień Piana
  [ 10] 0.5372  Zurada Sanibłysk
  [ 93] 0.5296  Zurada Gastro Moc


---
## Krok 4 — Filtr bezpieczeństwa LLM

Identyczny jak w lekcji 11 (`SAFETY_SYSTEM_2`).

In [7]:
SAFETY_SYSTEM_2 = """Jesteś ekspertem chemicznym oceniającym bezpieczeństwo środków czyszczących.

Otrzymasz strony produktów z bazy Zurada oraz pytanie klienta.

Twoje jedyne zadanie: zidentyfikuj produkty NIEBEZPIECZNE dla opisanego kontekstu — nie filtruj
pod kątem trafności czy dopasowania (to robi osobny krok).

Wyklucz produkt (zwróć jego product_id) TYLKO jeśli zachodzi jeden z poniższych warunków:
1. Jego sekcja "Odradzane powierzchnie" zawiera powierzchnię WPROST wymienioną w pytaniu klienta.
2. Skład/opis wskazuje na KONKRETNĄ, BEZPOŚREDNIĄ niezgodność chemiczną z kontekstem
   (przykłady: silny kwas na marmur lub kamień naturalny, podchloryn na aluminium lub tkaniny).
3. Ostrzeżenia BHP tego produktu wprost wykluczają opisany sposób użycia.

WAŻNE — produkty higieny osobistej i mydła:
Mydła w płynie, żele do mycia rąk, szampony i środki do higieny osobistej (kategorie vault:
'mydla-w-plynie', 'higiena-rak', 'higiena-osobista', 'mycie-rak') są z definicji bezpieczne
dla skóry i typowych powierzchni łazienkowych. Wyklucz je wyłącznie gdy kryterium 1 jest
spełnione — tzn. gdy ich "Odradzane powierzchnie" zawiera konkretną powierzchnię z pytania.
Nie wykluczaj ich na podstawie kryterium 2 ani 3.

ZASADY OGÓLNE:
- Ten filtr ocenia wyłącznie BEZPIECZEŃSTWO, nie trafność produktu.
- Nie wykluczaj produktu tylko dlatego, że jest silny, profesjonalny lub nie najlepiej pasuje.
- Gdy wątpliwość — zachowaj produkt (nie wykluczaj)."""


class SafetyFilterModel(BaseModel):
    excluded_ids: List[int] = Field(default_factory=list)
    reasoning: List[str]   = Field(default_factory=list)


def filter_unsafe(
    question: str,
    product_context: str,
    product_names: list[str],
) -> tuple[str, list[str], SafetyFilterModel, object]:
    """Remove unsafe products; returns (filtered_context, filtered_names, result, usage)."""
    safety, completion = llm.chat.completions.create_with_completion(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SAFETY_SYSTEM_2},
            {"role": "user",   "content": f"Strony produktów:\n\n{product_context}\n\n---\n\nPytanie klienta: {question}"},
        ],
        response_model=SafetyFilterModel,
        max_tokens=1000,
        temperature=0,
    )

    excluded_ids = set(safety.excluded_ids)
    id_map       = {p["product_id"]: p["name"] for p in products.values()}
    excluded_names = {id_map[pid] for pid in excluded_ids if pid in id_map}

    safe_names   = [n for n in product_names if n not in excluded_names]
    safe_context = load_product_pages(safe_names)
    return safe_context, safe_names, safety, completion.usage

---
## Krok 5 — Rekomendacja

Identyczna jak w lekcji 11 (`RECOMMEND_SYSTEM_3`).

In [8]:
RECOMMEND_SYSTEM_3 = """Jesteś ekspertem ds. środków czystości firmy Zurada.

{MAP_KNOWLEDGE}

Poniżej otrzymasz strony produktów, które przeszły już filtr bezpieczeństwa.
Opieraj się W 100% na faktach z dostarczonych stron — nie wymyślaj przeznaczenia produktu.

Oceń każdy produkt według 2 osi:

A) FORMA I TYP
   Forma (żel/piana/spray/proszek), przeznaczenie (ręczne vs maszynowe), przykłady ("np. X lub Y"
   to PRZYKŁADY — uwzględnij WSZYSTKIE produkty tego samego typu).

B) KONTEKST I SKALA
   Produkty profesjonalne rekomenduj gdy pasują. Wyklucz tylko czysto przemysłowe (beczki 200L).

Jeśli produkt ogólnie pasuje i nie ma przeciwwskazań — uwzględnij go.
Jeśli żaden nie pasuje — zwróć pustą listę product_ids."""


def recommend(question: str, product_context: str) -> tuple[RecommendModel, object]:
    result, completion = llm.chat.completions.create_with_completion(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": RECOMMEND_SYSTEM_3.format(MAP_KNOWLEDGE=MAP_KNOWLEDGE)},
            {"role": "user",   "content": f"Strony produktów (po filtrze bezpieczeństwa):\n\n{product_context}\n\n---\n\nPytanie klienta: {question}"},
        ],
        response_model=RecommendModel,
        max_tokens=4000,
        temperature=0,
    )
    return result, completion.usage

---
## Pełny pipeline

In [9]:
def qdrant_rag_pipeline(question: str) -> tuple[list[int], dict]:
    """intent → Qdrant retrieve → safety filter → recommend"""
    t0 = time.perf_counter()

    intent, usage_intent = extract_intent(question)

    product_context, product_names, scores = qdrant_retrieve(question, intent)

    safe_context, safe_names, safety_result, usage_safety = filter_unsafe(
        question, product_context, product_names
    )

    rec, usage_rec = recommend(question, safe_context)

    elapsed      = time.perf_counter() - t0
    returned_ids = [c.product_id for c in rec.product_ids]

    stats = {
        "intent":            {"surfaces": [s.value for s in intent.surfaces],
                              "categories": [c.value for c in intent.categories],
                              "methods": [m.value for m in intent.methods]},
        "n_retrieved":       len(product_names),
        "n_safe":            len(safe_names),
        "excluded_ids":      safety_result.excluded_ids,
        "top_score":         scores[0] if scores else None,
        "prompt_tokens":     usage_intent.prompt_tokens  + usage_safety.prompt_tokens  + usage_rec.prompt_tokens,
        "completion_tokens": usage_intent.completion_tokens + usage_safety.completion_tokens + usage_rec.completion_tokens,
        "total_tokens":      usage_intent.total_tokens   + usage_safety.total_tokens   + usage_rec.total_tokens,
        "time_seconds":      round(elapsed, 2),
    }
    return returned_ids, stats


# Single-run over all questions
results = []

for q in questions:
    qid      = q["id"]
    expected = set(q["expectedProductIds"])
    try:
        returned_ids, stats = qdrant_rag_pipeline(q["question"])
        returned = set(returned_ids)
        error    = None
    except Exception as e:
        returned = set()
        stats    = {"n_retrieved": 0, "n_safe": 0, "excluded_ids": [],
                    "top_score": None, "time_seconds": 0,
                    "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
        error    = str(e)

    correct = returned == expected
    results.append({
        "id": qid, "difficulty": q.get("difficulty", ""),
        "expected": sorted(expected), "returned": sorted(returned),
        "correct": correct,
        "missed":  sorted(expected - returned),
        "extra":   sorted(returned - expected),
        **{k: stats[k] for k in ["n_retrieved", "n_safe", "excluded_ids",
                                   "top_score", "total_tokens", "time_seconds"]},
        "intent": stats.get("intent", {}),
        "error":  error,
    })

    status = "✓" if correct else "✗"
    print(f"[{status}] Q{qid:02d} [{q['difficulty']:4s}]  "
          f"retrieved={stats['n_retrieved']:3d}→{stats['n_safe']:3d}  "
          f"top_score={stats['top_score'] or 0:.3f}  "
          f"tokens={stats['total_tokens']:6,}  "
          f"time={stats['time_seconds']:.1f}s  "
          f"expected={sorted(expected)}  returned={sorted(returned)}")

n_correct = sum(r["correct"] for r in results)
print(f"\n{n_correct}/{len(results)} exact matches  "
      f"avg time={sum(r['time_seconds'] for r in results)/len(results):.1f}s  "
      f"avg tokens={sum(r['total_tokens'] for r in results)/len(results):,.0f}")

[✗] Q01 [easy]  retrieved= 20→ 15  top_score=0.646  tokens=36,872  time=5.0s  expected=[12, 82, 139]  returned=[82, 139]
[✗] Q08 [easy]  retrieved= 20→  4  top_score=0.632  tokens=22,761  time=5.6s  expected=[136, 137, 189, 190]  returned=[136, 189, 190]
[✓] Q09 [easy]  retrieved=  8→  8  top_score=0.678  tokens=19,112  time=4.6s  expected=[233, 257]  returned=[233, 257]
[✓] Q10 [easy]  retrieved= 18→ 16  top_score=0.656  tokens=28,349  time=4.3s  expected=[213, 217, 218]  returned=[213, 217, 218]
[✗] Q02 [easy]  retrieved= 10→  8  top_score=0.634  tokens=24,069  time=13.0s  expected=[5, 34, 128]  returned=[5, 34, 96, 128]
[✗] Q03 [hard]  retrieved= 20→ 17  top_score=0.617  tokens=36,582  time=5.9s  expected=[45, 112, 152]  returned=[44, 122, 144]
[✓] Q04 [hard]  retrieved= 20→ 14  top_score=0.675  tokens=38,818  time=5.2s  expected=[11]  returned=[11]
[✗] Q05 [easy]  retrieved= 20→  1  top_score=0.682  tokens=26,177  time=6.2s  expected=[2, 39, 41, 129]  returned=[129]
[✓] Q06 [easy] 

---
## Zapis wyników (8 równoległych przebiegów)

In [11]:
N_RUNS      = 8
MAX_WORKERS = 9
OUTPUT_FILE = BASE_DIR / "qdrant_rag_runs.json"


def _run_once(q: dict, run_idx: int) -> tuple[int, int, dict]:
    try:
        returned_ids, stats = qdrant_rag_pipeline(q["question"])
        result = {
            "run":               run_idx,
            "returned_ids":      returned_ids,
            "model":             LLM_MODEL,
            "n_retrieved":       stats["n_retrieved"],
            "n_safe":            stats["n_safe"],
            "excluded_ids":      stats["excluded_ids"],
            "top_score":         stats["top_score"],
            "prompt_tokens":     stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "total_tokens":      stats["total_tokens"],
            "time_seconds":      stats["time_seconds"],
            "intent":            stats["intent"],
        }
    except Exception as e:
        result = {"run": run_idx, "error": str(e)}
    return q["id"], run_idx, result


tasks = [(q, run_idx) for q in questions for run_idx in range(1, N_RUNS + 1)]
raw: dict[int, list] = defaultdict(list)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(_run_once, q, run_idx): (q["id"], run_idx)
               for q, run_idx in tasks}
    for future in as_completed(futures):
        qid, run_idx, result = future.result()
        raw[qid].append(result)
        status = "✓" if not result.get("error") else "✗"
        print(f"  {status} Q{qid:02d} run {run_idx}")


repeated = []
for q in questions:
    qid  = q["id"]
    runs = sorted(raw[qid], key=lambda r: r["run"])

    times  = [r["time_seconds"] for r in runs if "time_seconds" in r]
    tokens = [r["total_tokens"]  for r in runs if "total_tokens"  in r]

    avg_time   = round(sum(times)  / len(times),  2) if times  else None
    avg_tokens = round(sum(tokens) / len(tokens), 0) if tokens else None

    repeated.append({
        "id":                   qid,
        "question":             q["question"],
        "expectedProductIds":   q["expectedProductIds"],
        "expectedNoProductIds": q["expectedNoProductIds"],
        "difficulty":           q.get("difficulty", ""),
        "avg_time_seconds":     avg_time,
        "avg_total_tokens":     avg_tokens,
        "runs":                 runs,
    })

    expected_set = set(q["expectedProductIds"])
    n_correct    = sum(1 for r in runs if set(r.get("returned_ids", [])) == expected_set)
    print(f"Q{qid:02d} [{q['difficulty']:4s}] {n_correct}/{N_RUNS} correct  "
          f"avg={avg_time:.1f}s  avg_tokens={avg_tokens:,.0f}")


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "llm_model":   LLM_MODEL,
        "embed_model": EMBED_MODEL,
        "approach":    "qdrant_rag",
        "n_runs":      N_RUNS,
        "questions":   repeated,
    }, f, ensure_ascii=False, indent=2)

print(f"\nSaved: {OUTPUT_FILE}")

  ✓ Q01 run 6
  ✓ Q01 run 8
  ✓ Q01 run 5
  ✓ Q08 run 1
  ✓ Q01 run 4
  ✓ Q01 run 1
  ✓ Q01 run 3
  ✓ Q01 run 7
  ✓ Q01 run 2
  ✓ Q08 run 2
  ✓ Q08 run 3
  ✓ Q09 run 1
  ✓ Q08 run 6
  ✓ Q08 run 4
  ✓ Q08 run 5
  ✓ Q09 run 2
  ✓ Q08 run 8
  ✓ Q08 run 7
  ✓ Q09 run 3
  ✓ Q09 run 4
  ✓ Q09 run 7
  ✓ Q09 run 6
  ✓ Q09 run 5
  ✓ Q09 run 8
  ✓ Q10 run 1
  ✓ Q10 run 2
  ✓ Q10 run 3
  ✓ Q10 run 4
  ✓ Q10 run 6
  ✓ Q10 run 8
  ✓ Q10 run 5
  ✓ Q02 run 1
  ✓ Q02 run 2
  ✓ Q10 run 7
  ✓ Q02 run 5
  ✓ Q02 run 4
  ✓ Q02 run 3
  ✓ Q02 run 7
  ✓ Q02 run 8
  ✓ Q02 run 6
  ✓ Q03 run 1
  ✓ Q03 run 3
  ✓ Q03 run 4
  ✓ Q03 run 2
  ✓ Q03 run 5
  ✓ Q03 run 6
  ✓ Q04 run 1
  ✓ Q03 run 7
  ✓ Q04 run 5
  ✓ Q04 run 6
  ✓ Q04 run 2
  ✓ Q03 run 8
  ✓ Q04 run 7
  ✓ Q05 run 3
  ✓ Q05 run 2
  ✓ Q05 run 4
  ✓ Q05 run 1
  ✓ Q04 run 8
  ✓ Q04 run 3
  ✓ Q04 run 4
  ✓ Q05 run 5
  ✓ Q05 run 6
  ✓ Q05 run 7
  ✓ Q06 run 1
  ✓ Q05 run 8
  ✓ Q06 run 2
  ✓ Q06 run 4
  ✓ Q06 run 7
  ✓ Q06 run 3
  ✓ Q06 run 6
  ✓ Q06 run 5
  ✓ Q0